# NB3 — Core-7 V2 × FashionCLIP embedding validation

NB3 validate ba lớp:

1. FashionCLIP cache;
2. versioned embedding manifest;
3. exact coverage của Core-7 V2 train/valid/test.

Report lưu SHA-256 của cache, manifest, positive JSONL và metadata JSONL. NB4 sẽ verify lại các hash này và hard-fail nếu report bị stale.

## 1. Runtime portable

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
AUTO_CLONE_REPO = False

def find_repo_root(start: Path = Path.cwd()):
    explicit = os.environ.get("FASHION_PROJECT_ROOT")
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
        raise FileNotFoundError(f"FASHION_PROJECT_ROOT không hợp lệ: {candidate}")
    current = start.expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root()
if REPO_ROOT is None and AUTO_CLONE_REPO:
    REPO_ROOT = (Path.cwd() / "opisoverated").resolve()
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
if REPO_ROOT is None:
    raise RuntimeError("Không tìm thấy repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.runtime_paths import load_runtime_paths
RUNTIME_PATHS = load_runtime_paths(repo_root=REPO_ROOT)
print("Repo root          :", REPO_ROOT)
print("Core-7 V2 folder :", RUNTIME_PATHS.core7_dir)
print("Embedding cache   :", RUNTIME_PATHS.embedding_cache)
print("Embedding manifest:", RUNTIME_PATHS.embedding_manifest)


## 2. Input artifacts

Embedding manifest là bắt buộc. Dùng `artifacts/embedding_manifest_v1.example.json` làm template rồi đặt manifest thực tế cạnh cache theo runtime config.

In [ ]:
CORE7_DIR = RUNTIME_PATHS.core7_dir
MAPPING_PATH = REPO_ROOT / "configs/category_mapping_core7_v2.json"
CACHE_PATH = RUNTIME_PATHS.embedding_cache
MANIFEST_PATH = RUNTIME_PATHS.embedding_manifest
SPLITS = ("train", "valid", "test")
POSITIVES = {split: CORE7_DIR / f"category_clean_{split}.jsonl" for split in SPLITS}
METADATA = {split: CORE7_DIR / f"core7_item_metadata_v1_{split}.jsonl" for split in SPLITS}
REPORT_PATH = CORE7_DIR / "core7_embedding_validation_report.json"

required = [MAPPING_PATH, CACHE_PATH, MANIFEST_PATH, *POSITIVES.values(), *METADATA.values()]
for path in required:
    print(path, "exists=", path.is_file())
    if not path.is_file():
        raise FileNotFoundError(path)


## 3. Validate cache + manifest + exact split coverage

In [ ]:
from src.data.validate_core7_embeddings import validate_core7_embedding_coverage

report = validate_core7_embedding_coverage(
    mapping_path=MAPPING_PATH,
    cache_path=CACHE_PATH,
    manifest_path=MANIFEST_PATH,
    positives_by_split=POSITIVES,
    metadata_by_split=METADATA,
    report_path=REPORT_PATH,
    expected_mapping_version="core7-v2",
)
print("Saved report:", REPORT_PATH)


## 4. Read gates

In [ ]:
print("CACHE PASS   :", report["cache"]["pass"])
print("MANIFEST PASS:", report["manifest"]["pass"])
print("MAPPING      :", report["category_mapping_version"])
print("EMBED VERSION:", report["embedding_version"])
for split in SPLITS:
    split_report = report["splits"][split]
    print(
        split,
        "pass=", split_report["pass"],
        "coverage=", f'{split_report["embedding_coverage"]:.4%}',
    )
print("OVERALL PASS               :", report["pass"])
print("READY FOR NEGATIVE SAMPLING:", report["ready_for_negative_sampling"])


## 5. Exact artifact fingerprints

Các SHA-256 dưới đây là identity của exact artifacts đã được NB3 validate. NB4 phải match chúng trước khi tạo negative.

In [ ]:
print("mapping sha256 :", report["inputs"]["category_mapping"]["sha256"])
print("base V1 sha256 :", report["inputs"]["category_mapping"]["base_sha256"])
print("resolved sha256: ", report["inputs"]["category_mapping"]["resolved_mapping_sha256"])
print("cache sha256   :", report["inputs"]["embedding_cache"]["sha256"])
print("manifest sha256:", report["inputs"]["embedding_manifest"]["sha256"])
for split in SPLITS:
    fingerprint = report["inputs"]["splits"][split]
    print(split, "positive=", fingerprint["positive_sha256"])
    print(split, "metadata=", fingerprint["metadata_sha256"])
